# SFT with QLoRA

Fine-tune Qwen3-8B on the reasoning traces using a rank-16 LoRA
adapter on top of a 4-bit quantized base model. Goal: teach the
model to produce DeepSeek-R1-style chain-of-thought before its
boxed answer. This is the most important training stage in the
whole pipeline — most of the GSM8K gain comes from here.

In [ ]:
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 170.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 70.5 MB/s eta 0:00:00


In [ ]:

!pip install -q -U bitsandbytes transformers trl peft accelerate datasets

# QLoRA setup. NF4 quantization with bf16 compute and double quant
# brings the 16GB base model down to ~5GB, leaving room for
# activations and optimizer state on a 40GB GPU.
# attn_implementation='sdpa' is the safe choice; switch to
# flash_attention_2 if you bothered to install flash-attn.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE = '/content/drive/MyDrive/llm_posttraining'

# Restore data from Drive to /content/
if not os.path.exists('/content/data/sft_train'):
    shutil.copytree(f'{DRIVE}/data/sft_train',    '/content/data/sft_train')
    shutil.copytree(f'{DRIVE}/data/sft_val',      '/content/data/sft_val')
    shutil.copytree(f'{DRIVE}/data/sft_test',     '/content/data/sft_test')
    shutil.copytree(f'{DRIVE}/data/grpo_prompts', '/content/data/grpo_prompts')
    print('Data restored.')
else:
    print('Data already present.')

Mounted at /content/drive
Data restored.


In [6]:
import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import wandb

# LoRA on all seven projection matrices in every transformer block.
# r=16, alpha=32 is the standard recipe for math fine-tuning. For
# an 80GB GPU r=32 alpha=64 is a small upgrade. Anything below r=8
# starts to hurt accuracy in our experience.
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL_NAME = 'Qwen/Qwen3-8B'
HF_REPO = 'Chaitanya77/qwen3-8b-math-sft'

print(f'VRAM: {VRAM_GB:.1f} GB')

VRAM: 102.0 GB


In [7]:
# Load the splits we saved in notebook 01. SFT only needs the
# formatted 'text' column; drop everything else to keep RAM low.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='sdpa', # this is the scaled dot product attention
)
model.config.use_cache = False
print('Base model loaded. VRAM:', round(torch.cuda.memory_allocated()/1e9, 2), 'GB')

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Base model loaded. VRAM: 9.68 GB


In [8]:
# Pick batch size and sequence length based on VRAM. Effective
# batch is held at 16 across both profiles so the learning rate
# doesn't need to change. paged_adamw_8bit is what makes the
# optimizer state fit at all on 40GB.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    # r=32 / alpha=64 if on 80GB and you want slightly stronger adaptation
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


In [9]:
# Optional but recommended: cap max_steps for faster iteration.
# 300 steps on H100 ≈ 2 hours and is enough to see a clear bump
# over baseline. Set packing=False since we don't have flash-attn —
# TRL will warn about cross-sample contamination otherwise.
train_ds = load_from_disk('/content/data/sft_train')
val_ds   = load_from_disk('/content/data/sft_val')

# SFT only needs the 'text' column
train_ds = train_ds.select_columns(['text'])
val_ds   = val_ds.select_columns(['text'])
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

Train: 30257 | Val: 3782


In [10]:
# Train. The trainer handles gradient checkpointing, evaluation,
# checkpointing, and W&B logging. Stop the cell early if Colab is
# about to disconnect — the latest checkpoint will still be on disk.
IS_80GB = VRAM_GB >= 75

sft_config = SFTConfig(
    output_dir='/content/checkpoints/sft',
    per_device_train_batch_size=4 if IS_80GB else 2,
    gradient_accumulation_steps=4 if IS_80GB else 8,
    max_steps=100,             # 1 epoch on 60k samples ~ 3-4h on A100 40GB
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=True,
    gradient_checkpointing=True,
    max_length=4096 if IS_80GB else 2048,
    dataset_text_field='text',
    optim='paged_adamw_8bit',       # paged optimizer for memory savings
    save_strategy='steps',
    save_steps=200,
    eval_strategy='steps',
    eval_steps=200,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='wandb',
    run_name='qwen3-8b-sft-math',
    packing=False,                   # packs short sequences for efficiency
)

wandb.init(project='verifiable-reasoning-posttraining', name='03_sft')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [11]:
train_ds = train_ds.select(range(5000))
val_ds = val_ds.select(range(500))

## Save the adapter

In [12]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)
trainer.train()
wandb.finish()

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.341487,0.343334,0.341905,4192783.000000,0.880027


wandb: WARNING URL not available in offline run


eval/entropy,▁
eval/loss,▁
eval/mean_token_accuracy,▁
eval/num_tokens,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/entropy,█▁
train/epoch,▁███
train/global_step,▁███
+5,...


## Quick eval after SFT

In [24]:
# Free the trainer before running lm-eval; otherwise we'd have two
# copies of the model in VRAM.
trainer.model.save_pretrained('/content/checkpoints/sft_final')
tokenizer.save_pretrained('/content/checkpoints/sft_final')
trainer.model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f'Adapter pushed to: https://huggingface.co/{HF_REPO}')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          | 1.21MB /  175MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp168ei43q/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Adapter pushed to: https://huggingface.co/Chaitanya77/qwen3-8b-math-sft


In [26]:
!pip install -q lm-eval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 12.6 MB/s eta 0:00:00


In [30]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.9 MB/s eta 0:00:00


In [34]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

!lm_eval \
    --model hf \
    --model_args pretrained=Qwen/Qwen3-8B,peft=/content/checkpoints/sft_final,dtype=bfloat16 \
    --tasks gsm8k_cot \
    --num_fewshot 8 \
    --apply_chat_template \
    --limit 100 \
    --batch_size 4 \
    --output_path /content/results/sft_gsm8k

2026-05-25:23:07:27 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-05-25:23:07:27 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-25:23:07:30 INFO     [_cli.run:388] Selected Tasks: ['gsm8k_cot']
2026-05-25:23:07:31 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-25:23:07:31 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen3-8B', 'peft': '/content/checkpoints/sft_final', 'dtype': 'bfloat16'}
2026-05-25:23:07:33 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-25:23:07:34 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.1

In [35]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Save SFT checkpoint
os.makedirs('/content/drive/MyDrive/llm_posttraining/checkpoints', exist_ok=True)
shutil.copytree('/content/checkpoints/sft_final',
                '/content/drive/MyDrive/llm_posttraining/checkpoints/sft_final',
                dirs_exist_ok=True)

# Save results
os.makedirs('/content/drive/MyDrive/llm_posttraining/results', exist_ok=True)
shutil.copytree('/content/results',
                '/content/drive/MyDrive/llm_posttraining/results',
                dirs_exist_ok=True)

print('Checkpoint and results saved to Drive.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint and results saved to Drive.
